# 15. DES-CPL: formulación, propiedades y complejidad

La guía del Entregable 3 (sección 5.4) pide, para el modelo original:
- la formulación matemática completa, con notación definida y supuestos explícitos;
- un pseudocódigo formal con entradas, salidas y pasos;
- al menos una propiedad enunciada y demostrada, o referenciada con precisión, con sus supuestos y su plausibilidad en los datos;
- el análisis de complejidad en notación $O(\cdot)$, contrastado con el tiempo empírico al variar $n$ y $p$.

Este capítulo lo reúne para **DES-CPL**. El capítulo 13 contó cómo se llegó al diseño y cómo se comportó; aquí se fija qué es exactamente y qué se puede garantizar de él. La implementación está en `src/des_cpl.py` y sus pruebas en `tests/test_des_cpl.py`.

In [1]:
from config import *
from IPython.display import display

from sklearn.metrics import average_precision_score

from src import experimento_des_cpl as exd
from src.des_cpl import (RegionCompetencia, competencia_global, competencias_locales,
                         combinar, perfil, pesos_softmax, podar, SUAVIZADO_PREVALENCIA)

CARPETA = RESULTADOS / "des_cpl_60"
seleccion = pd.read_csv(CARPETA / "seleccion_fold0.csv")
evaluado = pd.read_csv(CARPETA / "evaluacion_fold0.csv")
MODELOS = [c.removeprefix("oof_") for c in seleccion.columns if c.startswith("oof_")]
elegido = (pd.read_csv(RESULTADOS / "des_cpl_evaluacion_60.csv")
           .query("fold == 0 and método == 'DES-CPL'").iloc[0])
print(f"Fold 0: {len(seleccion)} encuentros de selección, {len(evaluado)} de evaluación, "
      f"{len(MODELOS)} modelos base")
print(f"Configuración elegida: k = {int(elegido['k'])}, τ = {elegido['tau']}, "
      f"λ = {elegido['lambda']}, modelos = {elegido['modelos_combinados']}")

Fold 0: 63579 encuentros de selección, 15894 de evaluación, 7 modelos base
Configuración elegida: k = 30, τ = 0.0001, λ = 1.0, modelos = logistica, random_forest, xgboost, svm


## 15.1. Notación y formulación

**Datos.**
- $\mathcal D = \{(x_i, y_i, g_i)\}_{i=1}^n$: encuentros de entrenamiento, con $y_i \in \{0, 1\}$ (readmisión en menos de 30 días) y $g_i$ el identificador del paciente.
- $\pi = \frac1n \sum_i y_i$: prevalencia global.
- $f_1, \dots, f_M$: clasificadores base probabilísticos, $f_m(x) \in (0, 1)$.

**1. Conjunto de selección fuera de pliegue.** Se parte $\mathcal D$ en $L = 3$ pliegues agrupados por paciente, $\mathcal D_1, \dots, \mathcal D_L$. Para el encuentro $i \in \mathcal D_\ell$,
$$\tilde f_m(x_i) = f_m^{(-\ell)}(x_i),$$
donde $f_m^{(-\ell)}$ se ajustó sin $\mathcal D_\ell$. Para predecir encuentros nuevos se usa $f_m$, ajustado con todo $\mathcal D$.

**2. Poda.** $\mathcal A \subseteq \{1, \dots, M\}$ son los $M' \in \{4, M\}$ modelos de mayor AUC-PR de $\tilde f_m$ en $\mathcal D$. A partir de aquí, $m$ recorre solo $\mathcal A$.

**3. Perfil de salida.** $\phi(x) = \big(\operatorname{logit} f_m(x)\big)_{m \in \mathcal A} \in \mathbb R^{M'}$, con las probabilidades recortadas a $[10^{-6}, 1 - 10^{-6}]$. Para los encuentros de selección se usa $\tilde f_m$ en lugar de $f_m$.

**4. Región de competencia.** Para una consulta $x$ (con paciente $g$ si pertenece a $\mathcal D$):
- $\mathcal N_k^{c}(x)$, con $c \in \{0, 1\}$: los $k$ encuentros de clase $c$ más cercanos a $\phi(x)$ en distancia euclídea, entre los que no son del paciente $g$;
- $\mathcal N_{2k}(x)$: los $2k$ más cercanos sin distinguir clase, con la misma exclusión.

**5. Prevalencia local suavizada**, con $s = 10$ pseudo-observaciones:
$$\hat q(x) = \frac{\sum_{i \in \mathcal N_{2k}(x)} y_i + s\,\pi}{2k + s}.$$

**6. Competencia local**: estimador estratificado del Brier del modelo $m$ en la zona.
$$\hat C_m^{\text{loc}}(x) = \hat q(x)\,\frac1k \sum_{i \in \mathcal N_k^{1}(x)} \big(\tilde f_m(x_i) - 1\big)^2 + \big(1 - \hat q(x)\big)\,\frac1k \sum_{i \in \mathcal N_k^{0}(x)} \tilde f_m(x_i)^2.$$

**7. Competencia global** (la misma medida, en todo el conjunto):
$$C_m^{\text{glob}} = \pi\,\overline{(\tilde f_m - 1)^2}\big|_{y=1} + (1 - \pi)\,\overline{\tilde f_m^{\,2}}\big|_{y=0} = \frac1n\sum_i \big(\tilde f_m(x_i) - y_i\big)^2.$$

**8. Encogimiento**, con $\lambda \in [0, 1]$:
$$\hat C_m(x) = (1 - \lambda)\,\hat C_m^{\text{loc}}(x) + \lambda\, C_m^{\text{glob}}.$$

**9. Pesos y predicción**, con temperatura $\tau \ge 0$:
$$w_m(x) = \frac{\exp\!\big(-\hat C_m(x)/\tau\big)}{\sum_{j \in \mathcal A} \exp\!\big(-\hat C_j(x)/\tau\big)}, \qquad \hat p(x) = \sum_{m \in \mathcal A} w_m(x)\, f_m(x).$$
El caso $\tau = 0$ se define como el límite: todo el peso, repartido por igual, va a los modelos de competencia mínima.

**Hiperparámetros.** $\theta = (M', k, \lambda, \tau)$ se eligen maximizando el AUC-PR de $\hat p$ sobre el propio conjunto de selección. En esa búsqueda la región de cada encuentro excluye a su paciente, y se usan 60 evaluaciones de Optuna (TPE) sobre la rejilla del protocolo.

**Supuestos.**
- **(S1)** Los pacientes son independientes entre sí; los encuentros de un mismo paciente pueden no serlo.
- **(S2)** Las predicciones fuera de pliegue $\tilde f_m$ se comportan como las de $f_m$ en datos nuevos: el modelo de cada pliegue se ajusta con 2/3 de los datos, lo que introduce un sesgo pequeño y conservador.
- **(S3)** La pérdida de Brier está acotada en $[0, 1]$.

## 15.2. Pseudocódigo

**Algoritmo 1. Ajuste de DES-CPL**

**Entradas:**
- encuentros $X$, etiquetas $y$ y pacientes $g$;
- modelos base $f_1, \dots, f_M$ con sus hiperparámetros;
- rejillas $\mathcal K$, $\Lambda$, $\mathcal T$ y $\{4, M\}$;
- presupuesto $B = 60$.

**Salida:** modelos base ajustados, conjunto de selección $\{(\tilde f(x_i), y_i, g_i)\}$, índices de vecinos y $\theta^\star = (M'^\star, k^\star, \lambda^\star, \tau^\star)$.

1. Partir $(X, y)$ en $L = 3$ pliegues estratificados y agrupados por $g$.
2. **para** cada modelo $m$ y cada pliegue $\ell$: ajustar $f_m^{(-\ell)}$ sin el pliegue $\ell$ y guardar $\tilde f_m(x_i) \leftarrow f_m^{(-\ell)}(x_i)$ para $i \in \mathcal D_\ell$.
3. **para** cada modelo $m$: ajustar $f_m$ con todo $(X, y)$.
4. **para** cada $M' \in \{4, M\}$:
   1. $\mathcal A_{M'} \leftarrow$ los $M'$ modelos de mayor AUC-PR fuera de pliegue.
   2. Construir los perfiles $\phi_i = \operatorname{logit} \tilde f_{\mathcal A}(x_i)$ y tres árboles $k$-d: positivos, negativos y todos.
   3. Calcular una sola vez, con $k_{\max} = \max \mathcal K$, los vecindarios $\mathcal N^1_{k_{\max}}$, $\mathcal N^0_{k_{\max}}$ y $\mathcal N_{2k_{\max}}$ de cada $i$, excluyendo los encuentros de su paciente $g_i$.
5. **repetir** $B$ veces (Optuna, TPE):
   1. Proponer $\theta = (M', k, \lambda, \tau)$.
   2. Recortar los vecindarios a $k$ y calcular $\hat q_i$, $\hat C^{\text{loc}}_{m}(x_i)$, $C^{\text{glob}}_m$ y $\hat C_m(x_i)$, que se guardan en caché por $(M', k)$.
   3. $\hat p_i \leftarrow \sum_m w_m(x_i)\, \tilde f_m(x_i)$ y puntaje $\leftarrow$ AUC-PR$(y, \hat p)$.
6. **devolver** el $\theta^\star$ de mayor puntaje.

**Algoritmo 2. Predicción de DES-CPL para una consulta $x$**

1. $p_m \leftarrow f_m(x)$ para $m \in \mathcal A_{M'^\star}$; $\phi \leftarrow \operatorname{logit}(p)$.
2. Buscar en los árboles $k$-d los vecindarios $\mathcal N^1_{k^\star}(x)$, $\mathcal N^0_{k^\star}(x)$ y $\mathcal N_{2k^\star}(x)$. La consulta no pertenece al conjunto de selección, así que no se excluye a nadie.
3. Calcular $\hat q(x)$, $\hat C_m(x)$ con $\lambda^\star$ y $w(x) = \operatorname{softmax}(-\hat C(x)/\tau^\star)$.
4. **devolver** $\hat p(x) = \sum_m w_m(x)\, p_m$.

**Criterio de parada:** el ajuste termina al agotar el presupuesto de $B = 60$ evaluaciones. No hay iteraciones internas: cada evaluación es un cálculo cerrado. En el código del capítulo 13 los vecindarios del fold de evaluación se calculan con los perfiles promediados de los modelos de pliegue. Así la región se busca en el mismo espacio en que se construyó el conjunto de selección.

## 15.3. Propiedades

### Proposición 1 (descomposición de ambigüedad del Brier)

Para cualquier $x$ y cualquier $y \in \{0, 1\}$, si $\hat p = \sum_m w_m p_m$ con $w_m \ge 0$ y $\sum_m w_m = 1$:
$$(\hat p - y)^2 = \sum_m w_m (p_m - y)^2 - \sum_m w_m (p_m - \hat p)^2.$$
En consecuencia, el Brier de DES-CPL en cada encuentro nunca supera el promedio ponderado (con sus propios pesos) del Brier de los modelos que combina. Solo lo iguala cuando todos los modelos dan la misma probabilidad.

*Demostración.* Se escribe $p_m - y = (p_m - \hat p) + (\hat p - y)$ y se eleva al cuadrado:
$$\sum_m w_m (p_m - y)^2 = \sum_m w_m (p_m - \hat p)^2 + 2(\hat p - y)\sum_m w_m (p_m - \hat p) + (\hat p - y)^2 \sum_m w_m.$$
El término cruzado se anula porque $\sum_m w_m p_m = \hat p$ y $\sum_m w_m = 1$. Despejando se obtiene la identidad. El segundo término es no negativo y vale cero si y solo si $p_m = \hat p$ para todo $m$ con $w_m > 0$. $\blacksquare$

**Alcance.** La identidad (Krogh y Vedelsby, 1995) vale para cualquier combinación convexa, así que no distingue a DES-CPL de un promedio. Lo que DES-CPL añade es la forma de elegir $w$: da más peso a los modelos con menor Brier estimado *en la zona de* $x$. Por eso la cota $\sum_m w_m (p_m - y)^2$ es, en esperanza local, menor que la del promedio simple cuando la competencia local se estima bien. Que se estime bien es justamente lo que estudian las proposiciones 2 a 4.

### Proposición 2 (la competencia por prevalencia es propia; la balanceada no)

Sea una zona del espacio de perfiles con riesgo $q = P(Y = 1 \mid \text{zona})$, y un modelo que predice la constante $p$ en ella.

- **(a)** La competencia reponderada por la prevalencia, $C_q(p) = q(1 - p)^2 + (1 - q)p^2$, es el Brier esperado en la zona y se minimiza en $p = q$: premia al modelo calibrado.
- **(b)** La competencia balanceada, $C_{1/2}(p) = \tfrac12(1 - p)^2 + \tfrac12 p^2$, se minimiza en $p = 1/2$ sea cual sea $q$.
- **(c)** Si, dentro de la zona, $\{\tilde f_m(x_i)\}_{i \in \mathcal N_k^1}$ y $\{\tilde f_m(x_i)\}_{i \in \mathcal N_k^0}$ son muestras independientes de las distribuciones condicionales a cada clase, y $q$ es conocida, entonces el estimador estratificado $\hat C_m^{\text{loc}}$ (con $q$ en lugar de $\hat q$) es insesgado para el Brier de la zona:
$$\mathbb E\big[\hat C_m^{\text{loc}}\big] = q\,B_1 + (1 - q)\,B_0 = \mathbb E\big[(f_m(X) - Y)^2 \mid \text{zona}\big], \quad B_c = \mathbb E\big[(f_m - c)^2 \mid Y = c, \text{zona}\big].$$
  Su varianza es $\big(q^2 \sigma_1^2 + (1 - q)^2 \sigma_0^2\big)/k$, donde $\sigma_c^2$ es la varianza de la pérdida dentro de la clase $c$.

*Demostración.*
- **(a)** $C_q'(p) = -2q(1 - p) + 2(1 - q)p = 2(p - q)$, que se anula en $p = q$. Como $C_q''(p) = 2 > 0$, es un mínimo. Además $C_q(p) = q(1 - q) + (p - q)^2$, así que el exceso de pérdida es exactamente el error cuadrático de calibración.
- **(b)** Es (a) con $q = 1/2$.
- **(c)** Por linealidad, $\mathbb E[\hat C] = q\,\mathbb E[\tfrac1k \sum_{\mathcal N^1}(\tilde f - 1)^2] + (1 - q)\,\mathbb E[\tfrac1k \sum_{\mathcal N^0}\tilde f^2] = q B_1 + (1 - q) B_0$, y por la ley de probabilidad total esto es el Brier de la zona. La varianza sale de la independencia de los $2k$ sumandos: $\operatorname{Var}\big(\tfrac{c}{k}\sum_{i=1}^k Z_i\big) = c^2 \sigma^2 / k$. $\blacksquare$

**Por qué importa.** (b) explica el fallo de la primera versión de DES-CPL (capítulo 13, sección 13.2). Con la mitad de vecinos positivos y peso $1/2$ por clase, la competencia premiaba al modelo cuya probabilidad estaba más cerca de 0.5. En este problema ese modelo es casi siempre el de probabilidades más altas, no el mejor calibrado.

(c) justifica estratificar: con la prevalencia del 11 % y $2k = 40$ vecinos sin estratificar, la probabilidad de que la región no tenga ningún positivo es $0.89^{40} \approx 1\,\%$, y la de que tenga menos de tres es cercana al 17 %. En esos casos $B_1$ no es estimable. La región estratificada garantiza $k$ positivos.

En la implementación, $q$ se reemplaza por $\hat q$, un estimador consistente (proposición 3). El estimador resultante es entonces asintóticamente insesgado, no exactamente insesgado.

### Proposición 3 (consistencia de la competencia local; resultado referenciado)

Bajo los supuestos (S1)-(S3), si el número de vecinos por clase cumple $k \to \infty$ y $k/n_c \to 0$ ($n_c$ es el número de encuentros de selección de la clase $c$), entonces, para casi todo $x$:
$$\hat C_m^{\text{loc}}(x) \xrightarrow{\;P\;} C_m(x) = q(x)\,B_{1,m}(x) + \big(1 - q(x)\big)\,B_{0,m}(x),$$
el Brier del modelo $m$ condicionado al perfil $\phi(x)$.

*Justificación.* Es el teorema de consistencia de los estimadores de vecinos más cercanos (Stone, 1977, teorema 1; Devroye, Györfi y Lugosi, 1996, cap. 6), aplicado tres veces:
- a la media de $(\tilde f_m - 1)^2$ entre los positivos, que converge a $B_{1,m}(x)$;
- a la media de $\tilde f_m^2$ entre los negativos, que converge a $B_{0,m}(x)$;
- a la proporción de positivos entre los $2k$ vecinos, que converge a $q(x)$; el suavizado con $s$ fijo desaparece cuando $k \to \infty$.

El teorema exige observaciones i.i.d. y una respuesta acotada; el Brier está en $[0, 1]$ por (S3). Las tres medias son funciones continuas de los límites, y la suma ponderada converge por el teorema de la función continua (Slutsky). $\blacksquare$

**Plausibilidad de los supuestos en estos datos.**
- **(S1)** Los encuentros de un mismo paciente comparten edad, diagnósticos e historia: el capítulo 3 midió $\eta^2 = 0.99$ en la edad entre encuentros del mismo paciente. Por eso los vecinos de un encuentro de selección excluyen a su propio paciente. Sin esa exclusión, el vecindario contendría casi copias de la consulta y la competencia mediría memoria, no generalización. Entre pacientes distintos se supone independencia, que es el supuesto estándar de la validación cruzada agrupada usada en todo el proyecto.
- **(S2)** Cada modelo de pliegue usa 2/3 de los datos. El costo es pequeño y medible: el capítulo 13 comparó $\tilde f_m$ con $f_m$ en el fold de evaluación.
- **Régimen asintótico.** El supuesto más débil es $k \to \infty$ con $k/n \to 0$. Con $n_1 \approx 7\,000$ positivos por fold, $k = 20$ es una fracción del 0.3 %, pero 20 sumandos dejan una varianza apreciable. Por eso existe el encogimiento (proposición 4).

### Proposición 4 (encogimiento óptimo hacia la competencia global)

Los pesos softmax no cambian si se suma la misma constante a las competencias de todos los modelos en $x$. Lo que cuenta, entonces, es la **competencia relativa**: $\Delta_m(x) = C_m(x) - \frac{1}{M'}\sum_j C_j(x)$. Como el encogimiento es lineal, $\hat\Delta_\lambda = (1 - \lambda)\hat\Delta^{\text{loc}} + \lambda\Delta^{\text{glob}}$.

Se fija $x$ y un modelo $m$, y se escribe $C = \Delta_m(x)$ para la competencia relativa verdadera, $\hat C^{\text{loc}}$ para su estimación local y $C^{\text{glob}}$ para la global. Se supone que:
- $\hat C^{\text{loc}}$ es (asintóticamente) insesgado, $\mathbb E[\hat C^{\text{loc}}] = C$, con varianza $v$;
- $C^{\text{glob}}$ es constante: se calcula con todo el conjunto, así que su varianza es despreciable frente a $v$;
- $b = C^{\text{glob}} - C$ es el sesgo de usar la competencia global en la zona de $x$, es decir, la heterogeneidad local.

Entonces el error cuadrático medio de $\hat C_\lambda = (1 - \lambda)\hat C^{\text{loc}} + \lambda C^{\text{glob}}$ es
$$\operatorname{ECM}(\lambda) = (1 - \lambda)^2\, v + \lambda^2\, b^2,$$
y se minimiza en
$$\lambda^\star = \frac{v}{v + b^2}.$$

*Demostración.* $\hat C_\lambda - C = (1 - \lambda)(\hat C^{\text{loc}} - C) + \lambda b$. Al tomar esperanza del cuadrado, el término cruzado $2\lambda(1 - \lambda)\, b\, \mathbb E[\hat C^{\text{loc}} - C]$ se anula por el insesgamiento. Queda $(1 - \lambda)^2 v + \lambda^2 b^2$, una parábola convexa en $\lambda$. Su derivada, $-2(1 - \lambda)v + 2\lambda b^2$, se anula en $\lambda^\star = v/(v + b^2) \in [0, 1]$. $\blacksquare$

**Lectura.** Es el razonamiento del estimador de James-Stein (Efron y Morris, 1973): se encoge hacia la media global en la medida en que el ruido local, $v$, domina sobre la heterogeneidad real, $b^2$.

La celda siguiente estima $v$ y $b^2$ en el fold 0 por el método de los momentos:
- $v$: varianza estratificada de (c) en la proposición 2, aplicada a la pérdida relativa de cada vecino (la de cada modelo menos la media de los modelos en ese vecino);
- $b^2 \approx \mathbb E\big[(\hat\Delta^{\text{loc}} - \Delta^{\text{glob}})^2\big] - v$, porque esa esperanza vale $b^2 + v$ bajo los supuestos anteriores.

Así se compara el $\lambda^\star$ que minimiza el error de *estimar la competencia* con el $\lambda$ que eligió la búsqueda, que maximiza el *AUC-PR*.

In [2]:
y_sel = seleccion["y"].to_numpy().astype(int)
g_sel = seleccion["grupo"].to_numpy()
P_sel = seleccion[[f"oof_{m}" for m in MODELOS]].to_numpy()

filas = []
for poda in (4, None):
    activos = podar(P_sel, y_sel, poda)
    Ps = P_sel[:, activos]
    region = RegionCompetencia(perfil(Ps), y_sel, g_sel)
    vecindario = region.vecindario(perfil(Ps), max(exd.K_REJILLA), g_sel)
    glob = competencia_global(Ps, y_sel)[0]
    for k in exd.K_REJILLA:
        c_loc, idx, q = competencias_locales(region, vecindario, Ps, k)
        q = np.asarray(q).reshape(-1, 1)
        # Competencias relativas: la softmax solo depende de las diferencias entre modelos.
        delta_loc = c_loc - c_loc.mean(axis=1, keepdims=True)
        delta_glob = glob - glob.mean()
        perdida_pos = (Ps[idx[:, k:]] - 1) ** 2          # (n, k, M') positivos
        perdida_neg = Ps[idx[:, :k]] ** 2                 # (n, k, M') negativos
        perdida_pos -= perdida_pos.mean(axis=2, keepdims=True)
        perdida_neg -= perdida_neg.mean(axis=2, keepdims=True)
        v = (q ** 2 * perdida_pos.var(axis=1, ddof=1)
             + (1 - q) ** 2 * perdida_neg.var(axis=1, ddof=1)) / k
        dispersion = (delta_loc - delta_glob) ** 2        # ≈ b² + v
        v_media, d_media = v.mean(), dispersion.mean()
        b2 = max(d_media - v_media, 0.0)
        filas.append({"modelos": len(activos), "k": k, "v (×10⁻⁶)": 1e6 * v_media,
                      "b² (×10⁻⁶)": 1e6 * b2, "λ* (ECM)": v_media / (v_media + b2)})
encogimiento = pd.DataFrame(filas)
guardar_resultado(encogimiento, "des_cpl_encogimiento_teorico")
display(encogimiento.round(3))
print(f"λ elegido por la búsqueda (AUC-PR) en el fold 0: {elegido['lambda']} "
      f"(k = {int(elegido['k'])}, {len(str(elegido['modelos_combinados']).split(','))} modelos)")

,modelos,k,v (×10⁻⁶),b² (×10⁻⁶),λ* (ECM)
0,4,5,0.895,17.704,0.048
1,4,10,0.601,9.860,0.057
2,4,20,0.405,5.517,0.068
3,4,30,0.323,4.289,0.070
4,4,50,0.237,3.062,0.072
5,4,100,0.149,1.969,0.070
6,7,5,5.505,2995.374,0.002
7,7,10,3.406,2669.569,0.001
8,7,20,2.132,2413.363,0.001
9,7,30,1.629,2298.857,0.001


λ elegido por la búsqueda (AUC-PR) en el fold 0: 1.0 (k = 30, 4 modelos)


**Lectura.** En el fold 0, el $\lambda^\star$ que minimiza el error de estimación es pequeño (del orden de 0.05 con 4 modelos y casi 0 con 7). Esto ocurre en todo $k$: la heterogeneidad $b^2$ supera en un orden de magnitud a la varianza $v$. Es decir, **la competencia relativa de los modelos sí cambia de una zona a otra**, y $k$ vecinos bastan para estimarla con precisión. Sin embargo, la búsqueda, que maximiza el AUC-PR, eligió $\lambda$ cercano a 1 en cuatro de los cinco folds.

Las dos cosas son compatibles. El Brier local mide la calidad de la probabilidad *dentro* de cada zona: qué modelo está mejor calibrado para pacientes de ese perfil. El AUC-PR, en cambio, mide el **orden entre pacientes de zonas distintas**. Usar pesos distintos en cada zona mejora el ajuste local, pero cambia la escala de $\hat p$ de una zona a otra, y eso puede desordenar pacientes que el promedio global ordenaba bien. La proposición 1 respalda esta lectura: lo que la ponderación local garantiza reducir es el Brier, no el ordenamiento.

Es el límite explicativo del capítulo 13 visto desde la teoría: hay diferencias locales entre los modelos, pero son diferencias de calibración, no de qué pacientes identifican como de mayor riesgo.

### Proposición 5 (casos límite de la temperatura)

Para $\hat C(x)$ fijo:
- si $\tau \to \infty$, entonces $w_m(x) \to 1/M'$ y $\hat p(x)$ tiende al promedio simple de los modelos podados;
- si $\tau \to 0^+$, entonces $w(x)$ tiende al reparto uniforme entre los modelos de competencia mínima: selección dinámica dura.

*Demostración.* Al multiplicar numerador y denominador por $\exp(\min_j \hat C_j/\tau)$ queda
$$w_m = \frac{\exp\big(-(\hat C_m - \min_j \hat C_j)/\tau\big)}{\sum_j \exp\big(-(\hat C_j - \min_i \hat C_i)/\tau\big)}.$$
- Con $\tau \to \infty$, cada exponente tiende a 0 y cada exponencial a 1.
- Con $\tau \to 0^+$, la exponencial vale 1 para los minimizadores y tiende a 0 para los demás.

$\blacksquare$

Junto con $\lambda$, esto muestra que DES-CPL **contiene como casos particulares** a los ensambles con que se comparó:
- $\lambda = 1$ con $\tau \to \infty$: promedio simple;
- $\lambda = 1$ con $\tau$ finito: pesos globales;
- $\lambda = 0$ con $\tau = 0$: selección dinámica clásica de un solo modelo.

Por eso, si la búsqueda recorriera la rejilla completa, DES-CPL nunca quedaría por debajo de esos ensambles **en el conjunto de selección**. Con 60 evaluaciones de Optuna la garantía es aproximada. Fuera del conjunto de selección sí puede quedar por debajo: es el costo de elegir entre más configuraciones, que el capítulo 13 midió.

## 15.4. Complejidad

Notación:
- $n$: encuentros de selección;
- $n_e$: consultas por predecir;
- $M$: modelos base; $M' \le M$: modelos podados, y dimensión del perfil;
- $k$: vecinos por clase, con $k_{\max} = \max \mathcal K$;
- $L = 3$: pliegues;
- $B = 60$: evaluaciones;
- $T_m(n)$ y $I_m(n)$: tiempo de ajuste y de predicción del modelo base $m$.

| Paso | Costo | Comentario |
|---|---|---|
| Conjunto de selección y modelos finales | $\sum_m \big[(L + 1)\,T_m(n) + I_m(n)\big]$ | Domina el costo total. Es el mismo para cualquier ensamble apilado |
| Poda | $O(M\, n \log n)$ | Un AUC-PR por modelo (ordenamiento) |
| Árboles $k$-d | $O(M'\, n \log n)$ | Tres árboles: positivos, negativos y todos |
| Vecindarios del conjunto de selección | $O\big(n\,(k_{\max} + r)\log n\big)$ en promedio | $r$: máximo de encuentros por paciente, que se piden de más para poder excluirlos. Válido en dimensión baja, $M' \le 7$ |
| Competencias para un par $(M', k)$ | $O(n\, k\, M')$ | Se calcula una vez por par y se guarda en caché |
| Búsqueda (B evaluaciones) | $O\big(|\mathcal K|\, n\, k_{\max} M' + B\, n \log n\big)$ | Cada evaluación, con la competencia en caché, es una softmax $O(nM')$ y un AUC-PR $O(n\log n)$ |
| **Predicción de $n_e$ consultas** | $O\big(\sum_m I_m(n_e) + n_e\,(k \log n + k M')\big)$ | Predicción de los modelos base, búsqueda de vecinos y competencia |

**Memoria.** $O(n\, k_{\max} M')$ para las pérdidas de la región. Es lo que obligó a submuestrear el conjunto CDC en la evaluación ampliada.

**Lo que sigue de la tabla.** El costo propio de DES-CPL (las filas 2 a 6) es casi lineal en $n$ y lineal en $M'$ y en $k$. Sobre él se suma el de reentrenar $L + 1 = 4$ veces cada modelo base. La celda siguiente lo contrasta con tiempos medidos en el fold 0:
- al variar $n$, submuestreando el conjunto de selección por paciente entre 4 000 y su tamaño completo;
- al variar la dimensión del perfil, $M' = 2, \dots, 7$.

En ambos casos se mide la construcción de los vecindarios, una evaluación de la competencia ($k = 20$) y la predicción de 10 000 consultas.

In [3]:
import time

def medir(P, y, g, consultas, k=20, k_max=max(exd.K_REJILLA)):
    t0 = time.perf_counter()
    region = RegionCompetencia(perfil(P), y, g)
    vec = region.vecindario(perfil(P), k_max, g)
    t1 = time.perf_counter()
    c, _, _ = competencias_locales(region, vec, P, k, encogimiento=0.5)
    combinar(pesos_softmax(c, 0.01), P)
    t2 = time.perf_counter()
    vec_q = region.vecindario(perfil(consultas), k)
    c_q, _, _ = competencias_locales(region, vec_q, P, k, encogimiento=0.5)
    combinar(pesos_softmax(c_q, 0.01), consultas)
    t3 = time.perf_counter()
    return {"vecindarios_s": t1 - t0, "competencia_s": t2 - t1, "prediccion_10k_s": t3 - t2}

rng = np.random.default_rng(RANDOM_STATE)
P_ev = evaluado[[f"final_{m}" for m in MODELOS]].to_numpy()
consultas = P_ev[rng.choice(len(P_ev), size=min(10_000, len(P_ev)), replace=False)]
pacientes = np.unique(g_sel)

filas = []
for n_objetivo in (4_000, 8_000, 16_000, 32_000, len(y_sel)):
    elegidos = rng.choice(pacientes, size=int(len(pacientes) * n_objetivo / len(y_sel)), replace=False)
    m = np.isin(g_sel, elegidos)
    filas.append({"variable": "n", "n": int(m.sum()), "M'": len(MODELOS),
                  **medir(P_sel[m], y_sel[m], g_sel[m], consultas)})
orden = np.argsort([-average_precision_score(y_sel, P_sel[:, j]) for j in range(len(MODELOS))])
for dim in range(2, len(MODELOS) + 1):
    activos = orden[:dim]
    filas.append({"variable": "M'", "n": len(y_sel), "M'": dim,
                  **medir(P_sel[:, activos], y_sel, g_sel, consultas[:, activos])})
escalamiento = pd.DataFrame(filas)
guardar_resultado(escalamiento, "des_cpl_escalamiento")
display(escalamiento.round(3))

def pendiente(tabla, x, y):
    return np.polyfit(np.log(tabla[x]), np.log(tabla[y]), 1)[0]

por_n = escalamiento.query("variable == 'n'")
por_m = escalamiento.query("variable == \"M'\"")
print("Exponente empírico en n (log-log):",
      {c: round(pendiente(por_n, "n", c), 2) for c in ["vecindarios_s", "competencia_s"]})
print("Exponente empírico en M':",
      {c: round(pendiente(por_m, "M'", c), 2) for c in ["vecindarios_s", "competencia_s"]})

tiempos = pd.read_csv(CARPETA / "tiempos.csv")
base = tiempos.groupby("modelo")[["t_pliegues_s", "t_final_s"]].mean()
print(f"\nCosto medio por fold de los modelos base (pliegues + final): {base.sum().sum() / 60:.1f} min; "
      f"vecindarios de DES-CPL con n completo: {por_n['vecindarios_s'].iloc[-1]:.1f} s")
display(base.round(1))

,variable,n,M',vecindarios_s,competencia_s,prediccion_10k_s
0,n,3932,7,0.507,0.038,0.412
1,n,7963,7,1.193,0.076,0.503
2,n,15989,7,2.938,0.160,0.626
3,n,31913,7,7.005,0.335,0.788
4,n,63579,7,21.049,0.747,1.270
5,M',63579,2,6.847,0.379,0.247
6,M',63579,3,8.799,0.445,0.419
7,M',63579,4,11.384,0.497,0.438
8,M',63579,5,14.093,0.556,0.665
9,M',63579,6,15.201,0.639,0.832


Exponente empírico en n (log-log): {'vecindarios_s': np.float64(1.33), 'competencia_s': np.float64(1.07)}
Exponente empírico en M': {'vecindarios_s': np.float64(0.85), 'competencia_s': np.float64(0.5)}

Costo medio por fold de los modelos base (pliegues + final): 5.3 min; vecindarios de DES-CPL con n completo: 21.0 s


,t_pliegues_s,t_final_s
modelo,,
arbol,11.5,5.4
bayes,9.9,4.1
knn,47.0,11.6
logistica,16.2,7.9
random_forest,55.3,27.9
svm,12.4,5.6
xgboost,59.2,41.4


**Lectura.** Las pendientes log-log contrastan la tabla de complejidad con la medición:
- **En $n$:** un exponente cercano a 1 (algo mayor por el factor $\log n$) confirma que la parte propia de DES-CPL crece casi linealmente con el número de encuentros.
- **En $M'$:** la competencia crece linealmente con la dimensión, pero los vecindarios dependen poco de ella mientras $M'$ sea pequeño, porque los árboles $k$-d funcionan bien en dimensión baja.
- **Costo total:** el del modelo lo domina el reentrenamiento de los modelos base, que es un orden de magnitud mayor que el de la región de competencia. DES-CPL es barato **dado** el conjunto de selección; lo caro es construir ese conjunto. Es el mismo costo que paga cualquier ensamble apilado con predicciones fuera de pliegue, y el protocolo lo declara por separado en la tabla de presupuesto.

**Referencias del capítulo.**
- Devroye, L., Györfi, L. y Lugosi, G. (1996). *A Probabilistic Theory of Pattern Recognition*. Springer.
- Efron, B. y Morris, C. (1973). Stein's estimation rule and its competitors: an empirical Bayes approach. *JASA*, 68(341), 117-130.
- Gneiting, T. y Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *JASA*, 102(477), 359-378.
- Krogh, A. y Vedelsby, J. (1995). Neural network ensembles, cross validation, and active learning. *NIPS 7*, 231-238.
- Stone, C. J. (1977). Consistent nonparametric regression. *The Annals of Statistics*, 5(4), 595-620.